In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [61]:
from pathlib import Path
import asyncio
from hedging_dataset_creator.hedging_labeller import hedging_labeller
from hedging_dataset_creator.label_transcript import label_transcript_sentences
from hedging_dataset_creator.sentence_tokenizer import sentence_tokenizer
from speech_parser.transcript_parser import parse_transcript_to_json
import pandas as pd

# Configuration variables
YEAR = "2017"
MONTH = "Jan"
DAY = "31"
COMPANY = "AMD"

TRANSCRIPT_PATH = f"data/Transcripts/{COMPANY}/{YEAR}-{MONTH}-{DAY}-{COMPANY}.txt"
PROCESSED_JSON_PATH = Path(f"data/processed/{YEAR}-{MONTH}-{DAY}-{COMPANY}.json")

parse_transcript_to_json(TRANSCRIPT_PATH)
sentences = pd.Series(sentence_tokenizer(PROCESSED_JSON_PATH)['sentence'])
sentences

0      Thank you and welcome to AMD's fourth-quarter ...
1                                  This is Laura Graves.
2      I recently joined the AMD IR team as Corporate...
3      By now you should have had the opportunity to ...
4      If you have not reviewed these documents, they...
                             ...                        
252                                 Thank you very much.
253                                 Thank you, operator.
254    Thank you, everyone, for joining us on our cal...
255    We look forward to speaking with you throughou...
256                                           Thank you.
Name: sentence, Length: 257, dtype: object

In [62]:
from hedging_dataset_creator.hedging_labeller import get_hedging_on_sentence
print(get_hedging_on_sentence(list(sentences[:10])))

You are a financial language expert specialising in earnings call analysis.

    Label the following sentences as hedging (1) or not hedging (0).

    Hedging language includes any of the following:
    - Modal verbs expressing uncertainty: may, might, could, would, should
    - Epistemic verbs: believe, think, expect, anticipate, feel, assume
    - Approximators: approximately, around, roughly, about
    - Probability expressions: likely, unlikely, possibly, probably, perhaps
    - Conditional framing: assuming, subject to, if, provided that
    - Opportunity hedges: potential, opportunity, possibility
    - Attribution hedges: based on current trends, according to our estimates
    - Distancing language: it appears, it seems, there is reason to believe
    - Negative hedges: cannot guarantee, no assurance, cannot predict
    
    Important:
    - Not every modal verb is hedging. "We will pay the dividend"
    is NOT hedging. "We believe we will pay the dividend" IS hedging.
    Direc

In [63]:
print('Processing', len(sentences), 'sentences through Anthropic')
#classified_sentences = await hedging_labeller(pd.DataFrame(sentences), semaphore=asyncio.Semaphore(10))
classified_sentences

Processing 257 sentences through Anthropic


Labelling hedging sentences: 100%|██████████| 26/26 [00:03<00:00,  7.22batch/s]


,sentence,isHedge
0,Thank you and welcome to AMD's fourth-quarter ...,0
1,This is Laura Graves.,0
2,I recently joined the AMD IR team as Corporate...,0
3,By now you should have had the opportunity to ...,1
4,"If you have not reviewed these documents, they...",1
...,...,...
252,Thank you very much.,0
253,"Thank you, operator.",0
254,"Thank you, everyone, for joining us on our cal...",0
255,We look forward to speaking with you throughou...,0


In [64]:
classified_sentences.to_csv(f"data/hedging_dataset/{COMPANY}/{YEAR}-{MONTH}-{DAY}-{COMPANY}.csv", index=False)

In [66]:
classified_sentences.groupby('isHedge')['sentence'].count()

isHedge
0    181
1     76
Name: sentence, dtype: int64

In [ ]:
temp = pd.read_csv('data/hedging_dataset/AMD/2016-Apr-21-AMD.csv')
temp.groupby('isHedge')['sentence'].count()


In [ ]:
amd_transcript_dir = Path("data/Transcripts/AMD")
transcript_files = sorted(amd_transcript_dir.glob("*.txt"))

print(f"Found {len(transcript_files)} AMD transcripts")
for transcript_file in transcript_files:
    print(f"Processing {transcript_file.name}")
    await label_transcript_sentences(str(transcript_file))

print("Done processing AMD transcripts")
